# Alternative Prediction Methods (ไม่ใช้ Time Series Lag)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akekapong78/ai-competition/blob/main/01-forecasting/alternative-methods/notebook.ipynb)

**XGBoost + lag features** คือ main approach แต่มีวิธีอื่นที่มีประโยชน์:

| Method | เหมาะกับสถานการณ์ไหน |
|--------|---------------------|
| **Linear Interpolation** | gap สั้น (< 3 ชม.) ไม่มี outlier |
| **Prophet** | ไม่อยากทำ feature engineering เลย |
| **KNN Regression** | หา "ชั่วโมงที่คล้ายกัน" จาก history |
| **MLP Neural Network** | เรียนรู้ non-linear pattern จาก features |
| **SVR** | dataset เล็ก (<10K rows) |

ท้ายสุด: **เปรียบเทียบทุก method** แล้วเลือกหรือ ensemble

In [ ]:
!pip install prophet scikit-learn xgboost -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.figsize'] = (13, 4)
print('Ready')

In [ ]:
# Dataset เดิม — synthetic solar
idx = pd.date_range('2023-01-01', '2024-12-31', freq='1h')
np.random.seed(42)
h = idx.hour
solar = np.maximum(0,
    10 * np.sin(np.pi*(h-6)/12)
    * (1 + 0.3*np.cos(2*np.pi*idx.month/12))
    + np.random.normal(0, 0.3, len(idx))
)
df = pd.DataFrame({'solar_mw': solar}, index=idx)

# สร้าง test set (ข้อมูลครึ่งหลังปี 2024 + จำลอง missing values)
test = df['2024-06-01':].copy()
train = df[:'2024-05-31'].copy()

# จำลอง missing values แบบ competition (missing หลายช่วง)
missing_mask = pd.Series(False, index=test.index)
for start_h in range(0, len(test), 48):   # ทุก 2 วัน
    gap_len = np.random.randint(1, 5)      # gap 1-4 ชั่วโมง
    missing_mask.iloc[start_h:start_h+gap_len] = True

test_with_gaps = test.copy()
test_with_gaps.loc[missing_mask, 'solar_mw'] = np.nan

print(f'Train: {len(train):,} rows | Test: {len(test):,} rows')
print(f'Missing: {missing_mask.sum()} rows ({missing_mask.mean()*100:.1f}%)')

---
## Method 1: Linear Interpolation (Baseline)

เติมค่าระหว่างจุดที่มีข้อมูล — เหมาะกับ gap สั้น

```
มีค่า: 5.2  ?  ?  ?  8.0
เติม:  5.2  6.05  6.9  7.45  8.0
```

**ข้อดี:** เร็วมาก, ไม่ต้อง train  
**ข้อเสีย:** ไม่รู้จัก pattern เวลา (เช่น solar ไม่ควรขึ้นตรงตอนกลางคืน)

In [ ]:
# Method 1: Linear Interpolation
interp_pred = test_with_gaps['solar_mw'].interpolate(method='linear')
interp_pred = interp_pred.clip(lower=0)  # solar ไม่ติดลบ

# Cubic Spline (smooth กว่า linear)
spline_pred = test_with_gaps['solar_mw'].interpolate(method='spline', order=3)
spline_pred = spline_pred.clip(lower=0)

mae_interp = mean_absolute_error(test.loc[missing_mask, 'solar_mw'], interp_pred[missing_mask])
mae_spline = mean_absolute_error(test.loc[missing_mask, 'solar_mw'], spline_pred[missing_mask])

print(f'Linear Interpolation MAE: {mae_interp:.3f} MW')
print(f'Spline Interpolation  MAE: {mae_spline:.3f} MW')

# Plot ตัวอย่าง gap
sample_start = test.index[48]
sample_end   = test.index[72]
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(test[sample_start:sample_end].index, test[sample_start:sample_end]['solar_mw'],
        'o-', label='Actual', color='orange', lw=2)
ax.plot(test_with_gaps[sample_start:sample_end].index,
        test_with_gaps[sample_start:sample_end]['solar_mw'],
        'x', label='Missing', color='red', ms=10, mew=2)
ax.plot(interp_pred[sample_start:sample_end].index, interp_pred[sample_start:sample_end],
        '--', label='Linear interp', color='blue')
ax.plot(spline_pred[sample_start:sample_end].index, spline_pred[sample_start:sample_end],
        ':', label='Spline interp', color='green', lw=2)
ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Interpolation Methods')
plt.tight_layout(); plt.show()

---
## Method 2: Prophet (Facebook/Meta)

Prophet ออกแบบมาสำหรับ business time series — **ไม่ต้องทำ feature engineering**

แค่บอกว่า:
- มี seasonality รายวัน / รายสัปดาห์ / รายปี
- มีวันหยุด (holiday)

Prophet จัดการให้เองหมด

```python
model = Prophet(daily_seasonality=True)
model.fit(df)
forecast = model.predict(future)
```

In [ ]:
from prophet import Prophet

# Prophet ต้องการ column ชื่อ 'ds' (datetime) และ 'y' (value)
train_prophet = train.reset_index().rename(columns={'index': 'ds', 'solar_mw': 'y'})

model_prophet = Prophet(
    daily_seasonality  = True,
    weekly_seasonality = True,
    yearly_seasonality = True,
    seasonality_mode   = 'multiplicative',  # solar เป็น multiplicative (กลางคืน = 0 เสมอ)
    changepoint_prior_scale = 0.05          # ความยืดหยุ่นของ trend
)
model_prophet.fit(train_prophet)

# Predict สำหรับ test period
future = pd.DataFrame({'ds': test.index})
forecast = model_prophet.predict(future)
prophet_pred = forecast.set_index('ds')['yhat'].clip(lower=0)

mae_prophet = mean_absolute_error(
    test.loc[missing_mask, 'solar_mw'],
    prophet_pred.loc[missing_mask]
)
print(f'Prophet MAE: {mae_prophet:.3f} MW')

In [ ]:
# Prophet components plot — ดูว่า model เข้าใจ pattern อะไรบ้าง
fig = model_prophet.plot_components(forecast)
plt.suptitle('Prophet Components: Trend + Seasonality', y=1.02)
plt.tight_layout()
plt.show()

---
## Method 3: KNN Regression

**ไอเดีย:** "ชั่วโมงที่คล้ายกันควรผลิตไฟได้ใกล้เคียงกัน"

หา K ชั่วโมงใน history ที่มี feature คล้ายที่สุด → average ค่า → ใช้เป็น prediction

```
Query: วันอังคาร เดือน 7 เวลา 10:00 น.

K=3 nearest neighbors:
  วันอังคาร เดือน 6 เวลา 10:00 → solar=8.2 MW
  วันพุธ    เดือน 7 เวลา 10:00 → solar=9.1 MW
  วันอังคาร เดือน 7 เวลา 10:00 (ปีที่แล้ว) → solar=8.7 MW

Prediction = mean(8.2, 9.1, 8.7) = 8.67 MW
```

**ข้อดี:** interpretable, ไม่ต้อง tune มาก  
**ข้อเสีย:** ช้าถ้า dataset ใหญ่, ต้องการ feature engineering

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

def time_features_only(df):
    """Features ที่ไม่ใช้ lag — ใช้ได้แม้ไม่มี history"""
    d = pd.DataFrame(index=df.index)
    d['hour']       = df.index.hour
    d['month']      = df.index.month
    d['dow']        = df.index.dayofweek
    d['hour_sin']   = np.sin(2*np.pi*df.index.hour/24)
    d['hour_cos']   = np.cos(2*np.pi*df.index.hour/24)
    d['month_sin']  = np.sin(2*np.pi*df.index.month/12)
    d['month_cos']  = np.cos(2*np.pi*df.index.month/12)
    d['solar_elev'] = np.maximum(0, np.sin(np.pi*(df.index.hour-6)/12))
    return d

X_train_knn = time_features_only(train)
y_train_knn = train['solar_mw']
X_test_knn  = time_features_only(test)

# Scale features — KNN sensitive ต่อ scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_knn)
X_test_scaled  = scaler.transform(X_test_knn)

knn = KNeighborsRegressor(n_neighbors=10, weights='distance', n_jobs=-1)
knn.fit(X_train_scaled, y_train_knn)
knn_pred = np.maximum(0, knn.predict(X_test_scaled))
knn_pred_s = pd.Series(knn_pred, index=test.index)

mae_knn = mean_absolute_error(test.loc[missing_mask, 'solar_mw'], knn_pred_s[missing_mask])
print(f'KNN (k=10) MAE: {mae_knn:.3f} MW')

---
## Method 4: MLP Neural Network

Neural Network ธรรมดา (ไม่ใช่ LSTM) — ใช้ time features เป็น input

```
Input layer:  [hour_sin, hour_cos, month_sin, month_cos, solar_elev, ...]
              ↓
Hidden layer 1: 128 neurons (ReLU)
              ↓
Hidden layer 2: 64 neurons (ReLU)
              ↓
Output layer:  1 neuron = predicted solar_mw
```

**ข้อดี:** จับ non-linear pattern ซับซ้อนได้  
**ข้อเสีย:** ต้อง tune มากกว่า, ช้ากว่า XGBoost บน CPU

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(
    hidden_layer_sizes = (128, 64),  # 2 hidden layers
    activation         = 'relu',
    solver             = 'adam',
    learning_rate_init = 0.001,
    max_iter           = 500,
    early_stopping     = True,       # หยุดเมื่อ val loss ไม่ลด
    validation_fraction= 0.1,
    random_state       = 42,
    verbose            = False
)

mlp.fit(X_train_scaled, y_train_knn)  # ใช้ same scaled features
mlp_pred = np.maximum(0, mlp.predict(X_test_scaled))
mlp_pred_s = pd.Series(mlp_pred, index=test.index)

mae_mlp = mean_absolute_error(test.loc[missing_mask, 'solar_mw'], mlp_pred_s[missing_mask])
print(f'MLP Neural Network MAE: {mae_mlp:.3f} MW')
print(f'Converged in {mlp.n_iter_} iterations')

---
## Method 5: SVR (Support Vector Regression)

หา "hyperplane" ที่ครอบ data ส่วนใหญ่ไว้ใน margin  
เหมาะกับ dataset เล็ก-กลาง ที่ noise ต่ำ

**Kernel trick:** แปลง features ไปมิติสูงกว่า → เส้นตรงในมิติสูง = เส้นโค้งในมิติเดิม

In [ ]:
from sklearn.svm import SVR

# SVR ช้ากับ data ใหญ่ — sample เฉพาะ 3000 rows สำหรับ demo
sample_n   = 3000
X_tr_small = X_train_scaled[:sample_n]
y_tr_small = y_train_knn.iloc[:sample_n]

svr = SVR(kernel='rbf', C=10, gamma='scale', epsilon=0.1)
svr.fit(X_tr_small, y_tr_small)
svr_pred = np.maximum(0, svr.predict(X_test_scaled))
svr_pred_s = pd.Series(svr_pred, index=test.index)

mae_svr = mean_absolute_error(test.loc[missing_mask, 'solar_mw'], svr_pred_s[missing_mask])
print(f'SVR (rbf kernel, 3K train sample) MAE: {mae_svr:.3f} MW')
print('หมายเหตุ: train ด้วย 3K rows เท่านั้น ไม่ใช่ full dataset')

---
## เปรียบเทียบทุก Method + Ensemble

In [ ]:
import xgboost as xgb

# XGBoost (เปรียบเทียบ) — ไม่ใช้ lag ในที่นี้เพื่อ fair comparison
xgb_model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                               subsample=0.8, random_state=42, n_jobs=-1, verbosity=0)
xgb_model.fit(X_train_scaled, y_train_knn)
xgb_pred_s = pd.Series(
    np.maximum(0, xgb_model.predict(X_test_scaled)),
    index=test.index
)
mae_xgb = mean_absolute_error(test.loc[missing_mask, 'solar_mw'], xgb_pred_s[missing_mask])

# Ensemble: average ของ methods ที่ดีที่สุด
ensemble_pred = (xgb_pred_s + knn_pred_s + prophet_pred + mlp_pred_s) / 4
mae_ens = mean_absolute_error(test.loc[missing_mask, 'solar_mw'], ensemble_pred[missing_mask])

# Summary table
results = {
    'Linear Interpolation': mae_interp,
    'Spline Interpolation':  mae_spline,
    'Prophet':               mae_prophet,
    'KNN (k=10)':            mae_knn,
    'MLP Neural Net':        mae_mlp,
    'SVR (3K sample)':       mae_svr,
    'XGBoost (no lag)':      mae_xgb,
    'Ensemble (avg 4)':      mae_ens,
}

res_df = pd.DataFrame(list(results.items()), columns=['Method', 'MAE (MW)'])
res_df = res_df.sort_values('MAE (MW)')

print('=' * 45)
print(f'  {"Method":<25} {"MAE (MW)":>10}')
print('-' * 45)
for _, row in res_df.iterrows():
    marker = ' ← best' if row['MAE (MW)'] == res_df['MAE (MW)'].min() else ''
    print(f'  {row["Method"]:<25} {row["MAE (MW)"]:>10.3f}{marker}')
print('=' * 45)

In [ ]:
# Bar chart เปรียบเทียบ
fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#2196F3' if v > res_df['MAE (MW)'].min() * 1.05 else '#4CAF50'
          for v in res_df['MAE (MW)']]
bars = ax.barh(res_df['Method'], res_df['MAE (MW)'], color=colors)
ax.set_xlabel('MAE (MW) — ยิ่งต่ำยิ่งดี')
ax.set_title('Method Comparison: MAE บน Missing Values')
for bar, val in zip(bars, res_df['MAE (MW)']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

In [ ]:
# Visual comparison กับ actual values
sample = slice('2024-07-01', '2024-07-05')
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(test[sample].index, test[sample]['solar_mw'],
        'o-', label='Actual', color='black', lw=2, ms=4)
ax.plot(prophet_pred[sample].index, prophet_pred[sample],
        '--', label='Prophet', color='purple', alpha=0.8)
ax.plot(knn_pred_s[sample].index, knn_pred_s[sample],
        ':', label='KNN', color='green', lw=2)
ax.plot(mlp_pred_s[sample].index, mlp_pred_s[sample],
        '-.', label='MLP', color='red', alpha=0.8)
ax.plot(ensemble_pred[sample].index, ensemble_pred[sample],
        '-', label='Ensemble', color='orange', lw=2.5)

ax.set_title('Prediction Comparison (5 วัน)')
ax.legend(ncol=3); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## เมื่อไหร่ควรใช้ Method ไหน?

```
gap < 3 ชั่วโมง + ไม่มี outlier
  → Linear Interpolation  (เร็ว, ง่าย)

มีเวลา train น้อย + ไม่อยากทำ feature engineering
  → Prophet  (ง่าย, decent accuracy)

Dataset เล็ก (< 5K rows)
  → KNN หรือ SVR

Dataset ใหญ่ + อยากอธิบาย model ได้ (pitching)
  → XGBoost + SHAP

ต้องการ accuracy สูงสุด
  → Ensemble: XGBoost (with lag) + LightGBM + Prophet
```

**Competition strategy:**  
train หลาย models → ensemble → ใช้ interpolation เป็น fallback สำหรับ gap สั้น